# PyHealth Metrics Tutorial

This notebook covers **`pyhealth.metrics`** — a collection of evaluation functions for clinical prediction tasks.

You will learn:
- **Binary classification** metrics: AUC-ROC, AUC-PR, F1, ECE, and more
- **Multiclass classification** metrics: accuracy, macro/micro F1
- **Multilabel classification** metrics: hamming loss, sample-level AUC
- **Fairness metrics**: disparate impact and statistical parity difference
- How to call `trainer.inference()` to get raw predictions for custom evaluation

> **Design note:** All metric functions in PyHealth accept raw numpy arrays, so they can be used independently of the Trainer or with any ML framework.

---

In [ ]:
import numpy as np
from pyhealth.metrics import (
    binary_metrics_fn,
    multiclass_metrics_fn,
    multilabel_metrics_fn,
)
from pyhealth.metrics.fairness import fairness_metrics_fn

# Set random seed for reproducibility
np.random.seed(42)

---
## Part 1: Binary Classification Metrics

Binary classification is the most common setup in clinical prediction:
- In-hospital mortality (alive / deceased)
- Readmission within 30 days (yes / no)
- Disease onset (positive / negative)

```python
binary_metrics_fn(
    y_true:    np.ndarray,              # shape (n_samples,), values in {0, 1}
    y_prob:    np.ndarray,              # shape (n_samples,), values in [0, 1]
    metrics:   Optional[List[str]] = None,  # default: ["pr_auc", "roc_auc", "f1"]
    threshold: float = 0.5,            # decision boundary for accuracy/F1/etc.
)
```

### Supported metrics

| Metric | Description |
|--------|-------------|
| `roc_auc` | Area under ROC curve — threshold-free measure of discrimination |
| `pr_auc` | Area under Precision-Recall curve — better for imbalanced datasets |
| `f1` | Harmonic mean of precision and recall |
| `accuracy` | Fraction of correct predictions |
| `balanced_accuracy` | Accuracy adjusted for class imbalance |
| `precision` | TP / (TP + FP) |
| `recall` | TP / (TP + FN) |
| `cohen_kappa` | Agreement beyond chance |
| `jaccard` | Intersection over union for positive class |
| `ECE` | Expected Calibration Error (calibration quality) |
| `ECE_adapt` | Adaptive ECE (equal-mass bins) |

In [ ]:
# --- Simulate a binary classification scenario: mortality prediction ---
# 300 patients, ~20% mortality rate (realistic for ICU cohorts)
n = 300
y_true_binary = np.random.binomial(1, 0.20, size=n).astype(np.float32)

# Simulate model probabilities: a reasonably well-calibrated model
# True positives get higher probabilities on average
y_prob_binary = np.where(
    y_true_binary == 1,
    np.random.beta(5, 2, size=n),   # higher probs for true positives
    np.random.beta(2, 5, size=n),   # lower probs for true negatives
).astype(np.float32)

print(f"Samples: {n}")
print(f"Positive rate: {y_true_binary.mean():.1%}")
print(f"Mean predicted prob (positives): {y_prob_binary[y_true_binary == 1].mean():.3f}")
print(f"Mean predicted prob (negatives): {y_prob_binary[y_true_binary == 0].mean():.3f}")

In [ ]:
# --- Default metrics ---
results_default = binary_metrics_fn(y_true_binary, y_prob_binary)
print("Default binary metrics:")
for metric, value in results_default.items():
    print(f"  {metric:15s}: {value:.4f}")

In [ ]:
# --- Full metric suite ---
results_full = binary_metrics_fn(
    y_true_binary,
    y_prob_binary,
    metrics=[
        "roc_auc",
        "pr_auc",
        "f1",
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall",
        "cohen_kappa",
        "jaccard",
        "ECE",
        "ECE_adapt",
    ],
    threshold=0.5,
)

print("Full binary metric suite (threshold=0.5):")
for metric, value in results_full.items():
    print(f"  {metric:20s}: {value:.4f}")

In [ ]:
# --- Impact of threshold on threshold-dependent metrics ---
print("Effect of decision threshold on F1 and precision/recall:")
print(f"  {'threshold':>10}  {'precision':>10}  {'recall':>10}  {'f1':>10}")

for threshold in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    m = binary_metrics_fn(
        y_true_binary, y_prob_binary,
        metrics=["precision", "recall", "f1"],
        threshold=threshold,
    )
    print(f"  {threshold:>10.1f}  {m['precision']:>10.4f}  {m['recall']:>10.4f}  {m['f1']:>10.4f}")

### When to use PR-AUC vs ROC-AUC

- **ROC-AUC** is appropriate when the classes are roughly balanced. It measures discrimination across all thresholds.
- **PR-AUC** is better for **imbalanced** datasets (like in-hospital mortality, where death is rare). It focuses on performance on the positive class and is not inflated by the large number of true negatives.

In clinical tasks with positive rates < 20%, always report PR-AUC alongside ROC-AUC.

---
## Part 2: Multiclass Classification Metrics

Multiclass is used when predicting among 3+ mutually exclusive outcomes, for example:
- Primary discharge diagnosis category (e.g., CCS groups)
- Length-of-stay bucket (short / medium / long)
- Triage acuity level (1–5)

```python
multiclass_metrics_fn(
    y_true:  np.ndarray,             # shape (n_samples,), integer class indices
    y_prob:  np.ndarray,             # shape (n_samples, num_classes), sum to 1
    metrics: Optional[List[str]] = None,  # default: ["accuracy", "f1_macro", "f1_micro"]
)
```

### Macro vs Micro averaging

| Averaging | Computes | Best for |
|-----------|----------|----------|
| `macro` | Mean of per-class metric | When all classes are equally important |
| `micro` | Global TP/FP/FN counts | When overall performance matters more than per-class balance |
| `weighted` | Weighted by class support | When you want to account for class frequency |

In [ ]:
# --- Simulate multiclass: 4 diagnostic categories ---
n_classes = 4
n_mc = 400
# Ground truth: somewhat imbalanced (class 0 is most common)
class_probs = [0.45, 0.25, 0.20, 0.10]
y_true_mc = np.random.choice(n_classes, size=n_mc, p=class_probs).astype(np.int64)

# Model outputs: softmax probabilities
# Simulate a decent model by perturbing a one-hot representation
y_prob_mc = np.zeros((n_mc, n_classes))
for i, label in enumerate(y_true_mc):
    probs = np.random.dirichlet([0.5] * n_classes)
    probs[label] += 1.5   # boost the true class
    probs /= probs.sum()
    y_prob_mc[i] = probs
y_prob_mc = y_prob_mc.astype(np.float32)

print(f"Samples: {n_mc}, Classes: {n_classes}")
print(f"Class distribution: {np.bincount(y_true_mc)}")

In [ ]:
# --- Default multiclass metrics ---
results_mc = multiclass_metrics_fn(y_true_mc, y_prob_mc)
print("Default multiclass metrics:")
for metric, value in results_mc.items():
    print(f"  {metric:20s}: {value:.4f}")

In [ ]:
# --- Extended multiclass metrics ---
results_mc_full = multiclass_metrics_fn(
    y_true_mc,
    y_prob_mc,
    metrics=[
        "accuracy",
        "f1_macro",
        "f1_micro",
        "f1_weighted",
        "precision_macro",
        "recall_macro",
        "roc_auc_macro_ovr",  # one-vs-rest ROC-AUC
        "ECE",
    ],
)

print("Extended multiclass metrics:")
for metric, value in results_mc_full.items():
    print(f"  {metric:25s}: {value:.4f}")

---
## Part 3: Multilabel Classification Metrics

Multilabel is used when each sample can have **multiple simultaneous labels**, for example:
- Drug recommendation: prescribe multiple drugs simultaneously
- Comorbidity prediction: patient can have several conditions
- Procedure recommendation: multiple procedures per visit

```python
multilabel_metrics_fn(
    y_true:    np.ndarray,              # shape (n_samples, n_labels), binary
    y_prob:    np.ndarray,              # shape (n_samples, n_labels), in [0, 1]
    metrics:   Optional[List[str]] = None,  # default: ["pr_auc_samples"]
    threshold: float = 0.3,             # note: lower default than binary!
)
```

Note the **threshold=0.3** default (vs 0.5 for binary). In drug recommendation, it is better to recommend slightly more drugs than to miss necessary ones.

In [ ]:
# --- Simulate multilabel: drug recommendation scenario ---
# 200 patients, 50 possible drugs, each patient prescribed 3-8 drugs on average
n_ml = 200
n_labels = 50

# True prescriptions: sparse binary matrix
y_true_ml = (np.random.rand(n_ml, n_labels) < 0.12).astype(np.float32)  # ~12% = ~6 drugs/patient

# Model probabilities
y_prob_ml = np.zeros((n_ml, n_labels), dtype=np.float32)
for i in range(n_ml):
    # True drugs get higher predicted probability
    y_prob_ml[i] = np.where(
        y_true_ml[i] == 1,
        np.random.beta(4, 2, size=n_labels),
        np.random.beta(1, 6, size=n_labels),
    )

print(f"Samples: {n_ml}, Labels: {n_labels}")
print(f"Mean labels per patient: {y_true_ml.sum(axis=1).mean():.1f}")

In [ ]:
# --- Default multilabel metrics ---
results_ml = multilabel_metrics_fn(y_true_ml, y_prob_ml)
print("Default multilabel metrics (threshold=0.3):")
for metric, value in results_ml.items():
    print(f"  {metric:25s}: {value:.4f}")

In [ ]:
# --- Extended multilabel metrics ---
results_ml_full = multilabel_metrics_fn(
    y_true_ml,
    y_prob_ml,
    metrics=[
        "pr_auc_samples",     # average PR-AUC per sample (most common)
        "pr_auc_macro",       # macro-averaged PR-AUC across labels
        "roc_auc_macro",      # macro-averaged ROC-AUC across labels
        "f1_macro",
        "f1_micro",
        "f1_samples",         # per-sample F1, then averaged
        "hamming_loss",       # fraction of label-sample pairs incorrectly classified
        "accuracy",           # exact match: all labels must be correct
    ],
    threshold=0.3,
)

print("Extended multilabel metrics:")
for metric, value in results_ml_full.items():
    print(f"  {metric:25s}: {value:.4f}")

### Interpreting multilabel metrics

| Metric | Clinical interpretation |
|--------|------------------------|
| `pr_auc_samples` | How well the model ranks drugs for each patient — the primary metric for drug recommendation |
| `hamming_loss` | Fraction of all (patient, drug) pairs incorrectly classified — penalizes false positives equally |
| `accuracy` (exact match) | Very strict — 1 only if all drugs are exactly correct |
| `f1_macro` | Per-drug average F1 — gives equal weight to rare and common drugs |

---
## Part 4: Fairness Metrics

Fairness metrics assess whether a model's performance is **equitable across subgroups** defined by sensitive attributes (e.g., race, sex, age group). This is crucial in clinical AI to avoid perpetuating historical health disparities.

```python
fairness_metrics_fn(
    y_true:               np.ndarray,   # (n_samples,) true labels
    y_prob:               np.ndarray,   # (n_samples,) predicted probabilities
    sensitive_attributes: np.ndarray,   # (n_samples,) 1=protected group, 0=unprotected
    favorable_outcome:    int = 1,      # which label value is considered positive
    metrics:              Optional[List[str]] = None,  # default: both below
    threshold:            float = 0.5,
)
```

### Supported fairness metrics

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| `disparate_impact` | P(ŷ=1 | protected) / P(ŷ=1 | unprotected) | Should be ≥ 0.8 (80% rule). 1.0 = perfect parity |
| `statistical_parity_difference` | P(ŷ=1 | protected) − P(ŷ=1 | unprotected) | Should be close to 0. Negative = protected group predicted positive less often |

In [ ]:
# --- Simulate a biased binary classifier ---
# Scenario: predicting ICU readmission
# Protected group (attr=1): elderly patients (age >= 65)
# Unprotected group (attr=0): younger patients (age < 65)

n_fair = 400
np.random.seed(99)

# 40% of patients are elderly
sensitive = np.random.binomial(1, 0.40, size=n_fair)  # 1 = elderly

# True outcomes — elderly have slightly higher readmission rate
y_true_fair = np.where(
    sensitive == 1,
    np.random.binomial(1, 0.35, size=n_fair),  # elderly: 35% readmission
    np.random.binomial(1, 0.25, size=n_fair),  # younger: 25% readmission
).astype(np.float32)

# Biased model: under-predicts readmission for elderly
# (e.g., trained on historical data that under-tested elderly patients)
y_prob_fair = np.where(
    sensitive == 1,
    np.random.beta(2, 5, size=n_fair),   # lower probs for elderly (biased)
    np.random.beta(3, 4, size=n_fair),   # higher probs for younger
).astype(np.float32)

print(f"Patients: {n_fair}")
print(f"Protected (elderly) group: {sensitive.sum()} ({sensitive.mean():.1%})")
print(f"True readmission rate — elderly:  {y_true_fair[sensitive==1].mean():.1%}")
print(f"True readmission rate — younger:  {y_true_fair[sensitive==0].mean():.1%}")
print(f"Mean predicted prob — elderly:    {y_prob_fair[sensitive==1].mean():.3f}")
print(f"Mean predicted prob — younger:    {y_prob_fair[sensitive==0].mean():.3f}")

In [ ]:
# --- Compute fairness metrics ---
fairness_results = fairness_metrics_fn(
    y_true=y_true_fair,
    y_prob=y_prob_fair,
    sensitive_attributes=sensitive,
    favorable_outcome=1,  # readmission = 1 is the "positive" outcome
    metrics=["disparate_impact", "statistical_parity_difference"],
    threshold=0.5,
)

print("Fairness metrics:")
for metric, value in fairness_results.items():
    print(f"  {metric:35s}: {value:.4f}")

In [ ]:
# --- Interpret the results ---
di  = fairness_results["disparate_impact"]
spd = fairness_results["statistical_parity_difference"]

print("Interpretation:")
print()
print(f"  Disparate Impact = {di:.4f}")
if di >= 0.8:
    print("    ✓ Above 0.8 threshold — model passes the 80% rule")
else:
    print("    ✗ Below 0.8 threshold — model fails the 80% rule (legally significant in US)")
print()
print(f"  Statistical Parity Difference = {spd:.4f}")
if abs(spd) < 0.05:
    print("    ✓ Close to 0 — model predictions are roughly equally distributed across groups")
elif spd < 0:
    print(f"    ✗ Negative ({spd:.4f}): protected group is predicted positive {abs(spd):.1%} less often")
else:
    print(f"    Protected group is predicted positive {spd:.1%} more often")

In [ ]:
# --- Compare against a fairer model (calibrated equally for both groups) ---
y_prob_fair2 = np.where(
    y_true_fair == 1,
    np.random.beta(4, 2, size=n_fair),
    np.random.beta(2, 4, size=n_fair),
).astype(np.float32)

fairness_results2 = fairness_metrics_fn(
    y_true=y_true_fair,
    y_prob=y_prob_fair2,
    sensitive_attributes=sensitive,
    threshold=0.5,
)

print("Fairer model fairness metrics:")
for metric, value in fairness_results2.items():
    print(f"  {metric:35s}: {value:.4f}")

### Clinical context for fairness metrics

In the healthcare domain, fairness is particularly important because:

1. **Historical bias in training data:** If a hospital historically provided less aggressive treatment to a subgroup, the training labels may reflect these disparities — and the model will learn to replicate them.

2. **Feature proxies:** Features like ZIP code, insurance type, or language can serve as proxies for race/ethnicity. A model may be technically race-unaware yet still exhibit disparate impact.

3. **Regulatory considerations:** The US 2021 Algorithmic Accountability Act and emerging EU AI Act both require documentation of bias audits for high-risk AI (which includes clinical decision support).

**PyHealth's approach:** Report fairness metrics alongside clinical performance metrics. A model with excellent AUC-ROC but poor disparate impact is not ready for deployment.

---
## Part 5: Integration with Trainer — Getting Raw Predictions

In practice you'll get predictions from a trained model and then compute any of the above metrics. The Trainer provides two ways:

**Option A — `trainer.evaluate(loader)`:** Returns a dict of metrics using the model's default metric function. Convenient, but limited to the default metrics.

**Option B — `trainer.inference(loader)`:** Returns raw numpy arrays `(y_true, y_prob, loss)`. Use this when you want to compute custom metrics, fairness analysis, or calibration plots.

```python
# After training (see tutorial_pyhealth_trainer.ipynb)
y_true, y_prob, loss = trainer.inference(test_loader)

# Now compute any metric combination you want
binary_metrics_fn(y_true, y_prob, metrics=["roc_auc", "pr_auc", "ECE"])
fairness_metrics_fn(y_true, y_prob, sensitive_attributes=race_labels)
```

The decoupling of **inference** from **metric computation** means you can:
- Cache the raw predictions and recompute metrics without re-running the model
- Apply post-hoc calibration (Platt scaling, temperature scaling) and re-evaluate
- Run bootstrap confidence intervals on any metric

---
## Summary

| Task type | Function | Default metrics |
|-----------|----------|-----------------|
| Binary | `binary_metrics_fn(y_true, y_prob)` | pr_auc, roc_auc, f1 |
| Multiclass | `multiclass_metrics_fn(y_true, y_prob)` | accuracy, f1_macro, f1_micro |
| Multilabel | `multilabel_metrics_fn(y_true, y_prob)` | pr_auc_samples |
| Fairness | `fairness_metrics_fn(y_true, y_prob, sensitive_attributes)` | disparate_impact, statistical_parity_difference |

### Choosing the right metric for clinical tasks

| Clinical Task | Recommended Primary Metric | Why |
|---------------|---------------------------|-----|
| Mortality prediction | `pr_auc` | Rare events (imbalanced); PR-AUC better than ROC-AUC |
| Readmission (30d) | `roc_auc` | Moderate prevalence; standard in literature |
| Drug recommendation | `pr_auc_samples` | Multi-label; need sample-level ranking quality |
| Disease severity (3+ levels) | `f1_macro` | Multi-class; equal weighting across severity levels |
| Any model in clinical use | + fairness metrics | Always audit for disparate impact |